# 05_parallel_and_loop

05_parallel_and_loop.py — ParallelAgent / LoopAgent 패턴 시연

ParallelAgent: 두 에이전트 동시 실행 (서로 다른 output_key 필수)
LoopAgent:     비평 → 개선을 만족할 때까지 반복. escalate 또는 max_iterations 로 종료.

⚠ 본 실습은 LLM 호출이 많아 free 모델 429 위험. 가벼운 구성만 시연.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
# 노트북 커널엔 이미 이벤트 루프가 돌아 스크립트의 asyncio.run() 이 깨짐 → nest_asyncio 로 중첩 허용
import nest_asyncio; nest_asyncio.apply()
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '05_parallel_and_loop.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
05_parallel_and_loop.py — ParallelAgent / LoopAgent 패턴 시연

ParallelAgent: 두 에이전트 동시 실행 (서로 다른 output_key 필수)
LoopAgent:     비평 → 개선을 만족할 때까지 반복. escalate 또는 max_iterations 로 종료.

⚠ 본 실습은 LLM 호출이 많아 free 모델 429 위험. 가벼운 구성만 시연.
"""
from google.adk.agents import LlmAgent, ParallelAgent, LoopAgent
from google.adk.tools.tool_context import ToolContext

from _adk_common import adk_model, run_once, banner, adk_unavailable


def exit_loop(tool_context: ToolContext):
    """비평 결과 더 고칠 게 없을 때 호출해 루프를 종료한다."""
    tool_context.actions.escalate = True
    return {"status": "exit"}


def main() -> None:
    banner("Parallel / Loop Workflow Agents — 골격 시연")

    # === ParallelAgent ===
    src_a = LlmAgent(
        name="src_a", model=adk_model(),
        instruction="주제에 대해 *학술적* 관점 한 문장만 출력. 다른 말 금지.",
        output_key="academic",
    )
    src_b = LlmAgent(
        name="src_b", model=adk_model(),
        instruction="주제에 대해 *실무적* 관점 한 문장만 출력. 다른 말 금지.",
        output_key="industry",      # 다른 키!
    )
    gather = ParallelAgent(name="gather", sub_agents=[src_a, src_b])

    # === LoopAgent (개념 정의만 — 실제 LLM 호출은 위 Parallel 만 시연) ===
    refiner = LlmAgent(
        name="refiner", model=adk_model(),
        instruction="초안 {draft} 를 한 줄 더 다듬어라.",
        output_key="draft",
    )
    critic = LlmAgent(
        name="critic", model=adk_model(),
        instruction=(
            "초안 {draft} 를 한 줄로 평가. 문제 없으면 exit_loop 호출."
        ),
        tools=[exit_loop],
    )
    refine_loop = LoopAgent(
        name="refine_loop",
        sub_agents=[refiner, critic],
        max_iterations=3,
    )

    print(f"\n  ✅ ParallelAgent 'gather' — sub_agents 2 개가 동시 실행")
    print(f"     각자 다른 output_key (academic / industry) 로 state 충돌 회피")
    print(f"\n  ✅ LoopAgent 'refine_loop' — max_iterations=3, escalate 종료")
    print(f"     critic 이 만족하면 exit_loop 도구로 escalate=True 신호")

    # 실제 실행 — Parallel 만 (LoopAgent 는 호출량이 커서 정의만 보여줌)
    print(f"\n  ▶ Parallel 실제 실행: '도구 호출 (function calling)' 주제")
    try:
        result = run_once(gather, "도구 호출 (function calling)")
        print(f"  💬 최종 응답 (병렬 에이전트는 final 답변 없을 수 있음):")
        print(f"     {result[:200]}")
        print(f"\n  → 두 sub_agent 가 state['academic'], state['industry'] 에 *동시* 기록")
    except Exception as e:
        print(f"  ⚠ {type(e).__name__}: {str(e)[:150]}")
        adk_unavailable()


if __name__ == "__main__":
    main()


📌 Parallel / Loop Workflow Agents — 골격 시연

  ✅ ParallelAgent 'gather' — sub_agents 2 개가 동시 실행
     각자 다른 output_key (academic / industry) 로 state 충돌 회피

  ✅ LoopAgent 'refine_loop' — max_iterations=3, escalate 종료
     critic 이 만족하면 exit_loop 도구로 escalate=True 신호

  ▶ Parallel 실제 실행: '도구 호출 (function calling)' 주제


C:\Users\user\AppData\Local\Temp\ipykernel_45556\2174589321.py:35: DeprecationWarning: ParallelAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  gather = ParallelAgent(name="gather", sub_agents=[src_a, src_b])
C:\Users\user\AppData\Local\Temp\ipykernel_45556\2174589321.py:50: DeprecationWarning: LoopAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  refine_loop = LoopAgent(



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



  💬 최종 응답 (병렬 에이전트는 final 답변 없을 수 있음):
     도구 호출(Function Calling)은 대규모 언어 모델이 외부 API나 정형화된 소프트웨어 도구와 상호작용하기 위해 구조화된 데이터를 생성함으로써, 모델의 추론 능력을 실행 가능한 외부 환경의 기능적 역량과 결합하는 메커니즘을 의미한다.

  → 두 sub_agent 가 state['academic'], state['industry'] 에 *동시* 기록
